# EDA: Web Robot Sessions

Mechanical descriptive EDA driven by `openbotrisk.eda`.

In [1]:
import sys, time, datetime as dt
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from openbotrisk.eda.loaders import load_web_robot_meta
from openbotrisk.eda.descriptive import missingness_table, cardinality_table, label_balance
from openbotrisk.eda.report import write_report

DATA_DIR = REPO_ROOT / 'data' / 'web-robot-sessions'
REPORT_PATH = REPO_ROOT / 'working' / 'eda' / 'web-robot-sessions.md'

t0 = time.time()
meta = load_web_robot_meta(DATA_DIR, json_sample_n=100)
elapsed = time.time() - t0
simple = meta['simple']
semantic = meta['semantic']
json_sample = meta['json_sample']
print(f'loaded in {elapsed:.1f}s; simple={simple.shape}, semantic={semantic.shape}, json_sample={len(json_sample)}')

loaded in 0.2s; simple=(67352, 32), semantic=(67352, 7), json_sample=100


In [2]:
def fmt_size(b):
    for u in ['B','KB','MB','GB']:
        if b < 1024: return f'{b:.1f} {u}'
        b /= 1024
    return f'{b:.1f} TB'

file_lines = '\n'.join(f'- `{n}` — {fmt_size(s)}' for n, s in sorted(meta['file_sizes'].items()))
today = dt.date.today().isoformat()
# The raw JSON sample shows the host `search.lib.auth.gr` (Aristotle University of Thessaloniki
# library OPAC) and timestamps in early 2018; the dataset matches Figshare 3477932
# (Rovetta et al., "Web Robot Detection" semantic + behavioural session features).
access = (
    'Source: Figshare dataset 3477932 (web robot detection, session-level features + raw HTTP logs).\n\n'
    'Underlying logs are from the Aristotle University of Thessaloniki library OPAC '
    '(`search.lib.auth.gr`), captured in early 2018 (per timestamps in `public_v2.json`).\n\n'
    f'Local path: `{meta["data_dir"]}`\n\n'
    'File formats: CSV (engineered session features) + JSON (raw per-request log).\n\n'
    f'Date inspected: {today}.\n\n'
    'Files on disk:\n' + file_lines
)
print(access)

Source: Figshare dataset 3477932 (web robot detection, session-level features + raw HTTP logs).

Underlying logs are from the Aristotle University of Thessaloniki library OPAC (`search.lib.auth.gr`), captured in early 2018 (per timestamps in `public_v2.json`).

Local path: `data/web-robot-sessions`

File formats: CSV (engineered session features) + JSON (raw per-request log).

Date inspected: 2026-05-23.

Files on disk:
- `public_v2.json` — 3.0 GB
- `semantic_features.csv` — 4.1 MB
- `simple_features.csv` — 15.2 MB


In [3]:
id_overlap = simple['ID'].isin(semantic['ID']).sum()
json_size_gb = meta['file_sizes'].get('public_v2.json', 0) / (1024**3)
structure = (
    f'- `simple_features.csv`: {len(simple):,} rows x {simple.shape[1]} cols. '
    'One row = one web session. HTTP/behavioural features + binary `ROBOT` label.\n'
    f'- `semantic_features.csv`: {len(semantic):,} rows x {semantic.shape[1]} cols. '
    'One row = one web session. Page-topic/semantic features + binary `ROBOT` label.\n'
    f'- `public_v2.json`: ~{json_size_gb:.1f} GB. Raw per-request log as a single JSON object '
    'keyed by request ID (Elasticsearch-style). Each value is a dict with HTTP request fields. '
    'Not loaded fully; only the first 100 entries were stream-parsed for schema inspection.\n'
    f'- Join key: `ID` links `simple_features` and `semantic_features` ({id_overlap:,} / {len(simple):,} '
    'session IDs overlap = 100%). The session `ID` is the parent key under which raw requests '
    'are grouped in the source logs; raw JSON uses request-level IDs (not session IDs) so direct '
    'session<->raw-request join would require an external mapping not included here.'
)
print(structure)

- `simple_features.csv`: 67,352 rows x 32 cols. One row = one web session. HTTP/behavioural features + binary `ROBOT` label.
- `semantic_features.csv`: 67,352 rows x 7 cols. One row = one web session. Page-topic/semantic features + binary `ROBOT` label.
- `public_v2.json`: ~3.0 GB. Raw per-request log as a single JSON object keyed by request ID (Elasticsearch-style). Each value is a dict with HTTP request fields. Not loaded fully; only the first 100 entries were stream-parsed for schema inspection.
- Join key: `ID` links `simple_features` and `semantic_features` (67,352 / 67,352 session IDs overlap = 100%). The session `ID` is the parent key under which raw requests are grouped in the source logs; raw JSON uses request-level IDs (not session IDs) so direct session<->raw-request join would require an external mapping not included here.


In [4]:
def schema_block(df, name, descs):
    rows = [f'### `{name}` ({df.shape[1]} columns)', '', '| column | dtype | example | description |', '|---|---|---|---|']
    samp = df.iloc[0].to_dict() if len(df) else {}
    for c in df.columns:
        v = samp.get(c, '')
        if pd.isna(v): v = ''
        v = str(v)[:40]
        d = descs.get(c, '(undocumented)')
        rows.append(f'| `{c}` | {df[c].dtype} | `{v}` | {d} |')
    return '\n'.join(rows)

simple_descs = {
    'ID': 'Session id (join key)',
    'NUMBER_OF_REQUESTS': 'Number of HTTP requests in session',
    'TOTAL_DURATION': 'Session duration (seconds)',
    'AVERAGE_TIME': 'Mean inter-request interval (s)',
    'STANDARD_DEVIATION': 'Std of inter-request interval (s)',
    'REPEATED_REQUESTS': 'Fraction of repeated resource requests',
    'HTTP_RESPONSE_2XX': 'Fraction of 2xx responses',
    'HTTP_RESPONSE_3XX': 'Fraction of 3xx responses',
    'HTTP_RESPONSE_4XX': 'Fraction of 4xx responses',
    'HTTP_RESPONSE_5XX': 'Fraction of 5xx responses',
    'GET_METHOD': 'Fraction of GET requests',
    'POST_METHOD': 'Fraction of POST requests',
    'HEAD_METHOD': 'Fraction of HEAD requests',
    'OTHER_METHOD': 'Fraction of other HTTP methods',
    'NIGHT': 'Fraction of requests during night hours',
    'UNASSIGNED': 'Fraction of requests with unassigned referrer',
    'IMAGES': 'Fraction of image resources',
    'TOTAL_HTML': 'Fraction of HTML resources',
    'HTML_TO_IMAGE': 'HTML-to-image request ratio',
    'HTML_TO_CSS': 'HTML-to-CSS request ratio',
    'HTML_TO_JS': 'HTML-to-JS request ratio',
    'WIDTH': 'Session navigation graph width',
    'DEPTH': 'Session navigation graph depth',
    'STD_DEPTH': 'Std of navigation depth',
    'CONSECUTIVE': 'Fraction of consecutive sequential requests',
    'DATA': 'Total bytes transferred',
    'PPI': 'Pages-per-interval (request rate proxy)',
    'SF_REFERRER': 'Same-frame referrer fraction',
    'SF_FILETYPE': 'Same-frame filetype fraction',
    'MAX_BARRAGE': 'Max burst size (consecutive rapid requests)',
    'PENALTY': 'Heuristic penalty score',
    'ROBOT': 'Target: 1 = bot, 0 = human',
}
semantic_descs = {
    'ID': 'Session id (join key)',
    'TOTAL_TOPICS': 'Total page topics visited',
    'UNIQUE_TOPICS': 'Distinct page topics visited',
    'PAGE_SIMILARITY': 'Mean pairwise page-content similarity',
    'PAGE_VARIANCE': 'Variance of page-content vectors',
    'BOOLEAN_PAGE_VARIANCE': 'Binary indicator of nontrivial page variance',
    'ROBOT': 'Target: 1 = bot, 0 = human',
}
json_descs = {
    'referrer': 'HTTP Referer header (URL or `-`)',
    'request': 'Full raw access-log request line',
    'method': 'HTTP method (GET/POST/HEAD/...)',
    'resource': 'Requested URL path',
    'bytes': 'Response size in bytes (string)',
    'response': 'HTTP status code (string)',
    'ip': 'Client IP (obfuscated to last octet jumbled in source)',
    'useragent': 'Client User-Agent string',
    'timestamp': 'ISO-8601 UTC timestamp of request',
}
schema_md = (
    schema_block(simple, 'simple_features', simple_descs)
    + '\n\n' + schema_block(semantic, 'semantic_features', semantic_descs)
    + '\n\n### `public_v2.json` (per-entry schema, from first 100 entries)\n\n'
    + '| field | type | example | description |\n|---|---|---|---|\n'
    + '\n'.join(
        f'| `{k}` | {meta["json_schema"].get(k, type(v).__name__)} | `{str(v)[:60].replace(chr(10)," ")}` | {json_descs.get(k, "")} |'
        for k, v in json_sample[0].items()
    )
    + '\n\nExample raw entry (formatted):\n\n'
    + '```json\n'
    + f'"{meta["json_sample_ids"][0]}": ' + str(json_sample[0])[:600]
    + '\n```'
)
print(schema_md[:600])

### `simple_features` (32 columns)

| column | dtype | example | description |
|---|---|---|---|
| `ID` | object | `obSnwGoBCue8G08E_WCX` | Session id (join key) |
| `NUMBER_OF_REQUESTS` | int64 | `79` | Number of HTTP requests in session |
| `TOTAL_DURATION` | int64 | `592` | Session duration (seconds) |
| `AVERAGE_TIME` | float64 | `7.5897436` | Mean inter-request interval (s) |
| `STANDARD_DEVIATION` | float64 | `1.8005404` | Std of inter-request interval (s) |
| `REPEATED_REQUESTS` | float64 | `0.0` | Fraction of repeated resource requests |
| `HTTP_RESPONSE_2XX` | float64 | `0.87341772151


In [5]:
lbl = label_balance(simple, 'ROBOT').sort_values('value')
lines = ['| ROBOT | count | rate |', '|---|---|---|']
for _, r in lbl.iterrows():
    lines.append(f'| {int(r["value"])} | {int(r["count"]):,} | {r["rate"]:.5f} |')
lbl_sem = label_balance(semantic, 'ROBOT').sort_values('value')
agree = (simple.set_index('ID')['ROBOT'] == semantic.set_index('ID')['ROBOT']).mean()
label_md = (
    'Label column: `ROBOT` in both `simple_features.csv` and `semantic_features.csv` '
    '(1 = bot, 0 = human).\n\n'
    + '\n'.join(lines) + '\n\n'
    f'Label agreement between the two CSVs on shared `ID`: {agree:.4f} '
    '(labels are derived from the same ground-truth session classification).\n\n'
    'Class imbalance: roughly 4:1 human:bot. Labels were assigned in the source dataset '
    'via heuristic + manual review of session user-agents and behaviour (per the Figshare/paper '
    'description); they are session-level rather than request-level.'
)
print(label_md)

Label column: `ROBOT` in both `simple_features.csv` and `semantic_features.csv` (1 = bot, 0 = human).

| ROBOT | count | rate |
|---|---|---|
| 0 | 53,858 | 0.79965 |
| 1 | 13,494 | 0.20035 |

Label agreement between the two CSVs on shared `ID`: 1.0000 (labels are derived from the same ground-truth session classification).

Class imbalance: roughly 4:1 human:bot. Labels were assigned in the source dataset via heuristic + manual review of session user-agents and behaviour (per the Figshare/paper description); they are session-level rather than request-level.


In [6]:
ct = cardinality_table(simple, ['ID'])
# Identifiers from raw JSON sample
sample_ips = set(s.get('ip') for s in json_sample if s.get('ip'))
sample_uas = set(s.get('useragent') for s in json_sample if s.get('useragent'))
lines = ['| column | source | n_unique (in scope) | role |', '|---|---|---|---|']
lines.append(f'| `ID` | both CSVs | {int(ct.iloc[0]["n_unique"]):,} | session primary key (Elasticsearch-style id) |')
lines.append(f'| `ip` | `public_v2.json` (per request) | {len(sample_ips)} (in {len(json_sample)}-sample) | client IP (obfuscated, weak actor id) |')
lines.append(f'| `useragent` | `public_v2.json` (per request) | {len(sample_uas)} (in {len(json_sample)}-sample) | UA string (weak actor/bot signal) |')
lines.append('| `referrer` | `public_v2.json` (per request) | n/a | referring URL |')
ident_md = (
    'The CSVs expose only the session `ID` as an identifier; per-session demographic / '
    'actor attributes are not present. Actor-level signals (IP, User-Agent) live in the '
    'raw `public_v2.json` log at the request level.\n\n'
    + '\n'.join(lines) + '\n\n'
    f'Example IPs from the JSON sample: `{", ".join(list(sample_ips)[:3])}` (note the source '
    'obfuscates the final octet of each IPv4 address by digit-jumbling).\n'
    f'Example User-Agents (truncated): {"; ".join(list(sample_uas)[i][:60] for i in range(min(3, len(sample_uas))))}.'
)
print(ident_md)

The CSVs expose only the session `ID` as an identifier; per-session demographic / actor attributes are not present. Actor-level signals (IP, User-Agent) live in the raw `public_v2.json` log at the request level.

| column | source | n_unique (in scope) | role |
|---|---|---|---|
| `ID` | both CSVs | 67,352 | session primary key (Elasticsearch-style id) |
| `ip` | `public_v2.json` (per request) | 9 (in 100-sample) | client IP (obfuscated, weak actor id) |
| `useragent` | `public_v2.json` (per request) | 7 (in 100-sample) | UA string (weak actor/bot signal) |
| `referrer` | `public_v2.json` (per request) | n/a | referring URL |

Example IPs from the JSON sample: `94.66.32960, 79.103.40554, 66.249.34457` (note the source obfuscates the final octet of each IPv4 address by digit-jumbling).
Example User-Agents (truncated): Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36; BUbiNG (+http://law.di.unimi.it/BUbiNG.html); ICC-Crawler/2.0 (Mozilla-compatible; ; http://ucri.nict.go.j.


In [7]:
# Temporal info lives only in the raw JSON timestamps (per request).
ts = [s.get('timestamp') for s in json_sample if s.get('timestamp')]
ts_parsed = pd.to_datetime(ts, errors='coerce', utc=True)
t_min = ts_parsed.min()
t_max = ts_parsed.max()
temporal_md = (
    'The CSVs contain only aggregated session-level temporal *features* (`TOTAL_DURATION`, '
    '`AVERAGE_TIME`, `STANDARD_DEVIATION`, `NIGHT`, `MAX_BARRAGE`); no wall-clock session '
    'start/end timestamps are exposed.\n\n'
    'Wall-clock timestamps live only in `public_v2.json` at the per-request level.\n\n'
    f'- Format: ISO-8601 UTC string, e.g. `{ts[0]}` (millisecond precision, `Z` suffix).\n'
    f'- Sample range ({len(ts)} requests from the head of the file): {t_min} to {t_max}.\n'
    '- The raw access-log line inside `request` also contains the original local timestamp '
    'with `+0200` offset (Athens local time), confirming the source is European.\n'
    '- Full file temporal range cannot be reported without scanning the 3 GB JSON, which is '
    'out of scope for this bounded EDA.'
)
print(temporal_md)

The CSVs contain only aggregated session-level temporal *features* (`TOTAL_DURATION`, `AVERAGE_TIME`, `STANDARD_DEVIATION`, `NIGHT`, `MAX_BARRAGE`); no wall-clock session start/end timestamps are exposed.

Wall-clock timestamps live only in `public_v2.json` at the per-request level.

- Format: ISO-8601 UTC string, e.g. `2018-02-28T22:00:01.000Z` (millisecond precision, `Z` suffix).
- Sample range (100 requests from the head of the file): 2018-02-28 22:00:01+00:00 to 2018-02-28 22:00:25+00:00.
- The raw access-log line inside `request` also contains the original local timestamp with `+0200` offset (Athens local time), confirming the source is European.
- Full file temporal range cannot be reported without scanning the 3 GB JSON, which is out of scope for this bounded EDA.


In [8]:
mt_s = missingness_table(simple)
mt_m = missingness_table(semantic)
hi_s = mt_s[mt_s['null_rate'] > 0.0]
hi_m = mt_m[mt_m['null_rate'] > 0.0]
missing_md = (
    f'- `simple_features`: {int(simple.isna().sum().sum()):,} null cells overall '
    f'({simple.isna().sum().sum() / simple.size:.4%} of cells). '
    f'{len(hi_s)} of {simple.shape[1]} columns have any nulls.\n'
    f'- `semantic_features`: {int(semantic.isna().sum().sum()):,} null cells overall '
    f'({semantic.isna().sum().sum() / semantic.size:.4%}). '
    f'{len(hi_m)} of {semantic.shape[1]} columns have any nulls.\n'
    f'- `public_v2.json`: per-entry schema is dense in the 100-row sample; '
    '`referrer` is encoded as the literal string `-` when absent (not JSON null), so a '
    'full-file null check would have to look for `-` sentinels rather than missing keys.\n\n'
    'Columns with any nulls in `simple_features`:\n\n'
    + ('| column | null_rate |\n|---|---|\n'
       + '\n'.join(f'| `{r["column"]}` | {r["null_rate"]:.6f} |' for _, r in hi_s.iterrows())
       if len(hi_s) else '_(none)_')
    + '\n\nColumns with any nulls in `semantic_features`:\n\n'
    + ('| column | null_rate |\n|---|---|\n'
       + '\n'.join(f'| `{r["column"]}` | {r["null_rate"]:.6f} |' for _, r in hi_m.iterrows())
       if len(hi_m) else '_(none)_')
)
print(missing_md)

- `simple_features`: 43,221 null cells overall (2.0054% of cells). 3 of 32 columns have any nulls.
- `semantic_features`: 26,328 null cells overall (5.5843%). 3 of 7 columns have any nulls.
- `public_v2.json`: per-entry schema is dense in the 100-row sample; `referrer` is encoded as the literal string `-` when absent (not JSON null), so a full-file null check would have to look for `-` sentinels rather than missing keys.

Columns with any nulls in `simple_features`:

| column | null_rate |
|---|---|
| `STANDARD_DEVIATION` | 0.213906 |
| `SF_FILETYPE` | 0.213906 |
| `SF_REFERRER` | 0.213906 |

Columns with any nulls in `semantic_features`:

| column | null_rate |
|---|---|
| `BOOLEAN_PAGE_VARIANCE` | 0.130301 |
| `PAGE_VARIANCE` | 0.130301 |
| `PAGE_SIMILARITY` | 0.130301 |


In [9]:
quirks_md = (
    '- Three-file layout: two session-feature CSVs (engineered) + one 3 GB raw JSON log. '
    'The CSVs are pre-computed features; modelling can use them directly without touching the raw log.\n'
    '- Both CSVs have identical row counts and a 100% overlap on `ID`; they are effectively two '
    'feature blocks for the same session table and can be inner-joined on `ID`.\n'
    '- The raw JSON is a single top-level object (one outer `{...}`) rather than NDJSON. Streaming '
    'parse is feasible because each entry happens to be written on its own line as `"<id>":{...},`, '
    'but this layout is fragile - any reformatter would break naive line-parsers.\n'
    '- Raw-log IDs are per-*request*, not per-*session*; the CSV `ID` is per-session. There is no '
    'in-file mapping from session ID to its constituent request IDs.\n'
    '- Client IPs in the raw log are partially obfuscated (the final octet is digit-shuffled, e.g. '
    '`66.249.34457`), so they cannot be geolocated or joined to external IP lists.\n'
    '- The `referrer` field uses `"-"` for missing values (Apache-log convention) instead of JSON null.\n'
    '- `PENALTY` in `simple_features` looks like a heuristic bot-score; if it was used to derive `ROBOT` '
    'it would leak the label. Verify before using as a feature.\n'
    '- Class balance ~80/20 human/bot - mild imbalance, manageable without resampling.\n'
    '- Source host (`search.lib.auth.gr`) is a single library OPAC, so traffic patterns and bot mix '
    'are domain-specific (academic search crawlers like Googlebot, ICC-Crawler dominate the bot class).'
)
print(quirks_md)

- Three-file layout: two session-feature CSVs (engineered) + one 3 GB raw JSON log. The CSVs are pre-computed features; modelling can use them directly without touching the raw log.
- Both CSVs have identical row counts and a 100% overlap on `ID`; they are effectively two feature blocks for the same session table and can be inner-joined on `ID`.
- The raw JSON is a single top-level object (one outer `{...}`) rather than NDJSON. Streaming parse is feasible because each entry happens to be written on its own line as `"<id>":{...},`, but this layout is fragile - any reformatter would break naive line-parsers.
- Raw-log IDs are per-*request*, not per-*session*; the CSV `ID` is per-session. There is no in-file mapping from session ID to its constituent request IDs.
- Client IPs in the raw log are partially obfuscated (the final octet is digit-shuffled, e.g. `66.249.34457`), so they cannot be geolocated or joined to external IP lists.
- The `referrer` field uses `"-"` for missing values (Apa

In [10]:
repro_md = (
    'Generated by `notebooks/eda/web-robot-sessions.ipynb` which calls '
    '`openbotrisk.eda.loaders.load_web_robot_meta` (pandas full-read for the two CSVs; '
    'manual line-streaming for the first 100 entries of `public_v2.json`).\n\n'
    'Run with:\n\n'
    '```bash\n'
    'jupyter nbconvert --to notebook --execute --inplace \\\n'
    '  notebooks/eda/web-robot-sessions.ipynb \\\n'
    '  --ExecutePreprocessor.timeout=300\n'
    '```\n\n'
    f'Loader runtime on this machine: {elapsed:.1f}s. The two CSVs ({simple.shape[0]:,} rows each) '
    'fit in memory; the 3 GB JSON is never fully materialised - only the first 100 entries are read.'
)
print(repro_md)

Generated by `notebooks/eda/web-robot-sessions.ipynb` which calls `openbotrisk.eda.loaders.load_web_robot_meta` (pandas full-read for the two CSVs; manual line-streaming for the first 100 entries of `public_v2.json`).

Run with:

```bash
jupyter nbconvert --to notebook --execute --inplace \
  notebooks/eda/web-robot-sessions.ipynb \
  --ExecutePreprocessor.timeout=300
```

Loader runtime on this machine: 0.2s. The two CSVs (67,352 rows each) fit in memory; the 3 GB JSON is never fully materialised - only the first 100 entries are read.


In [11]:
stats = {
    'Access': access,
    'Structure': structure,
    'Schema': schema_md,
    'Label': label_md,
    'Identifier inventory': ident_md,
    'Temporal structure': temporal_md,
    'Missing data': missing_md,
    'Quirks and observations': quirks_md,
    'Reproduction': repro_md,
}
out = write_report(stats, 'Web Robot Sessions', REPORT_PATH)
print('wrote', out)

wrote working/eda/web-robot-sessions.md
